# Assignment 3: Forward and backpropagation, learning rate and epochs
Part A implements forward + backward pass by hand with TensorFlow ops and verifies the gradients against `tf.GradientTape`. Part B analyses learning rate and epochs with Keras.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(0); np.random.seed(0)
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
sc = StandardScaler().fit(X_train)
X_train, X_test = sc.transform(X_train).astype("float32"), sc.transform(X_test).astype("float32")
Y_train = tf.one_hot(y_train, 3)

## Part A: manual forward and backward propagation
Network: `X -> Dense(8, ReLU) -> Dense(3, softmax)`, loss = categorical cross-entropy.

Forward: `z1 = XW1 + b1`, `a1 = relu(z1)`, `z2 = a1W2 + b2`, `p = softmax(z2)`

Backward (chain rule): `dz2 = (p - y)/N`, `dW2 = a1^T dz2`, `db2 = sum(dz2)`, `da1 = dz2 W2^T`, `dz1 = da1 * (z1>0)`, `dW1 = X^T dz1`, `db1 = sum(dz1)`

In [ ]:
def init_params(n_in=4, n_h=8, n_out=3, seed=0):
    g = tf.random.Generator.from_seed(seed)
    return {
        "W1": tf.Variable(g.normal([n_in, n_h], stddev=0.5)), "b1": tf.Variable(tf.zeros([n_h])),
        "W2": tf.Variable(g.normal([n_h, n_out], stddev=0.5)), "b2": tf.Variable(tf.zeros([n_out])),
    }

def forward(X, P):
    z1 = X @ P["W1"] + P["b1"]
    a1 = tf.nn.relu(z1)
    z2 = a1 @ P["W2"] + P["b2"]
    p = tf.nn.softmax(z2)
    return z1, a1, p

def loss_fn(p, Y):
    return -tf.reduce_mean(tf.reduce_sum(Y * tf.math.log(p + 1e-9), axis=1))

def backward(X, Y, P, z1, a1, p):
    N = tf.cast(tf.shape(X)[0], tf.float32)
    dz2 = (p - Y) / N
    dW2 = tf.transpose(a1) @ dz2
    db2 = tf.reduce_sum(dz2, axis=0)
    da1 = dz2 @ tf.transpose(P["W2"])
    dz1 = da1 * tf.cast(z1 > 0, tf.float32)
    dW1 = tf.transpose(X) @ dz1
    db1 = tf.reduce_sum(dz1, axis=0)
    return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2}

In [ ]:
# Gradient check: manual backprop vs automatic differentiation
P = init_params()
Xt = tf.constant(X_train)
z1, a1, p = forward(Xt, P)
manual = backward(Xt, Y_train, P, z1, a1, p)

with tf.GradientTape() as tape:
    _, _, p2 = forward(Xt, P)
    L = loss_fn(p2, Y_train)
auto = tape.gradient(L, P)

for k in P:
    diff = float(tf.reduce_max(tf.abs(manual[k] - auto[k])))
    print(f"{k}: max |manual - autodiff| = {diff:.2e}")

In [ ]:
def train_manual(lr=0.1, epochs=200):
    P = init_params()
    Xt = tf.constant(X_train)
    losses, accs = [], []
    for ep in range(epochs):
        z1, a1, p = forward(Xt, P)
        g = backward(Xt, Y_train, P, z1, a1, p)
        for k in P:
            P[k].assign_sub(lr * g[k])
        losses.append(float(loss_fn(p, Y_train)))
        accs.append(float(np.mean(np.argmax(p.numpy(), 1) == y_train)))
    return P, losses, accs

P, losses, accs = train_manual(0.1, 200)
_, _, pt = forward(tf.constant(X_test), P)
print("Manual-backprop test accuracy:", np.mean(np.argmax(pt.numpy(), 1) == y_test))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(losses); ax[0].set_title("Loss (manual backprop)")
ax[1].plot(accs); ax[1].set_title("Train accuracy"); plt.show()

## Part B: effect of learning rate and epochs (Keras)

In [ ]:
def make_model(lr):
    m = keras.Sequential([keras.layers.Input((4,)), keras.layers.Dense(8, activation="relu"), keras.layers.Dense(3, activation="softmax")])
    m.compile(optimizer=keras.optimizers.SGD(lr), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

lrs = [0.0001, 0.001, 0.01, 0.1, 1.0]
histories = {}
for lr in lrs:
    tf.keras.utils.set_random_seed(0)
    m = make_model(lr)
    h = m.fit(X_train, y_train, epochs=200, batch_size=16, validation_data=(X_test, y_test), verbose=0)
    histories[lr] = h.history

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for lr, h in histories.items():
    ax[0].plot(h["loss"], label=f"lr={lr}"); ax[1].plot(h["val_accuracy"], label=f"lr={lr}")
ax[0].set_title("Training loss vs epochs"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].set_title("Validation accuracy vs epochs"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

In [ ]:
# Grid: learning rate x number of epochs -> test accuracy
epoch_list = [10, 25, 50, 100, 200]
grid = pd.DataFrame(index=[f"lr={l}" for l in lrs], columns=[f"{e} ep" for e in epoch_list], dtype=float)
for lr in lrs:
    for e in epoch_list:
        tf.keras.utils.set_random_seed(0)
        m = make_model(lr)
        m.fit(X_train, y_train, epochs=e, batch_size=16, verbose=0)
        grid.loc[f"lr={lr}", f"{e} ep"] = m.evaluate(X_test, y_test, verbose=0)[1]
print(grid.round(3))

import seaborn as sns
plt.figure(figsize=(7, 4)); sns.heatmap(grid.astype(float), annot=True, fmt=".2f", cmap="viridis")
plt.title("Test accuracy: learning rate vs epochs"); plt.show()

## Observations
- **Very small lr (1e-4, 1e-3):** loss decreases slowly, model is under-trained even after 200 epochs.
- **Moderate lr (0.01-0.1):** fast and stable convergence, best accuracy.
- **Very large lr (1.0):** updates overshoot, loss can oscillate or diverge.
- **Epochs:** more epochs help small learning rates catch up; with a good lr, accuracy saturates and extra epochs add little (risk of overfitting).